# Testing Greedy Convayor & Greedy Point Map

This script will run two basic frameworks:
1. Greedy Algorithm which does X sweeps across the full SOM in a random order and moves the nodes in a SOM to their maximum value
2. Greedy RL Point Map which an RL picks and x,y and a greedy reward picks the node bin. The episodes start short and for every time an episode has 70% helpful answers the episode gets longer. Length of the episode is used to track progress

In [1]:
from __future__ import annotations
import numpy as np
from dataclasses import dataclass, asdict
from typing import Literal, Optional, Tuple, Dict, List
import matplotlib.pyplot as plt

from SOM_Reward_Test_Final import SOMFastTC
from envs import SOM, GlobalPointerTwoStageEnv, ConveyorSweepEnv

### None RL Methods: 



In [2]:
# -------------------------------------------------------------------
# 0) Utilities
# -------------------------------------------------------------------

@dataclass
class RunStats:
    initial_trace: float
    final_trace: float
    total_delta: float
    steps_taken: int
    positive_steps: int
    zero_steps: int
    negative_steps: int
    skipped_steps: int
    edits_applied: int

    def as_dict(self): return asdict(self)


def _summarize(trace_hist: List[float], deltas: List[float], applied: int, skipped: int) -> RunStats:
    initial = float(trace_hist[0])
    final = float(trace_hist[-1])
    total_delta = final - initial
    pos = int(np.sum(np.array(deltas) > 0))
    zero = int(np.sum(np.isclose(deltas, 0)))
    neg = int(np.sum(np.array(deltas) < 0))
    return RunStats(
        initial_trace=initial,
        final_trace=final,
        total_delta=total_delta,
        steps_taken=len(deltas),
        positive_steps=pos,
        zero_steps=zero,
        negative_steps=neg,
        skipped_steps=skipped,
        edits_applied=applied,
    )


# -------------------------------------------------------------------
# 1) Greedy Conveyor baseline (ordered sweep, multi-pass)
# -------------------------------------------------------------------

def greedy_conveyor(
    som: "SOM",
    passes: int = 2,
    order: Literal["random", "row_major", "hilbert"] = "random",
    top_M_bins: int = 3,
    allow_negative: bool = False,
    cooldown_steps: int = 12,
    seed: Optional[int] = 0,
) -> Tuple[np.ndarray, RunStats]:
    """
    Deterministic sweep over nodes (order fixed per run).
    For each node:
      - compute Δ for all bins via SOM.estimate_delta_trace_bins_for_node(node)
      - choose argmax Δ if Δ>0 (greedy). If allow_negative=True, take argmax even if ≤0
      - otherwise skip (mask) this node for `cooldown_steps` future encounters.

    Returns:
      final_bins (flat), stats
    """
    rng = np.random.default_rng(seed)
    H, W = som.grid_shape
    N = H * W
    K = som.num_bins

    # Build order
    if order == "row_major":
        idx_order = np.arange(N, dtype=np.int64)
    elif order == "random":
        idx_order = rng.permutation(N).astype(np.int64)
    else:
        # Minimal Hilbert fallback: use row_major but randomly rotate/flip grid to emulate locality changes
        grid = np.arange(N, dtype=np.int64).reshape(H, W)
        if rng.random() < 0.5: grid = np.flipud(grid)
        if rng.random() < 0.5: grid = np.fliplr(grid)
        k = rng.integers(0, 4)
        idx_order = np.rot90(grid, k).ravel()

    # Snapshot, run on a working copy
    base_state = som.get_state()
    trace_hist, deltas = [], []
    cool_until = np.zeros((N,), dtype=np.int64)
    edits_applied = 0
    skipped = 0

    t_abs = 0
    trace_hist.append(som.total_reward_fast())

    for p in range(passes):
        for ptr in range(N):
            node = int(idx_order[ptr])
            # cooldown?
            if cool_until[node] > t_abs:
                deltas.append(0.0); skipped += 1; t_abs += 1
                continue

            # greedy Δ over bins (absolute-bin)
            all_d = som.estimate_delta_trace_bins_for_node(node)
            cur_bin = int(som.get_assigned_bins(grid=False)[node])
            all_d[cur_bin] = -np.inf  # no-op forbidden

            b_star = int(np.argmax(all_d))
            d_star = float(all_d[b_star])

            if (d_star > 0.0) or allow_negative:
                # apply
                r0 = som.total_reward_fast()
                x, y = divmod(node, W)
                som.set_node_bin(x, y, b_star)
                r1 = som.total_reward_fast()
                delta = float(r1 - r0)
                deltas.append(delta)
                edits_applied += 1
                # simple tabu: if still negative in reality, cool this node
                if delta <= 0.0:
                    cool_until[node] = t_abs + cooldown_steps
            else:
                # skip & cool
                deltas.append(0.0)
                cool_until[node] = t_abs + cooldown_steps
                skipped += 1

            t_abs += 1
            trace_hist.append(som.total_reward_fast())

    final_bins = som.get_state()
    stats = _summarize(trace_hist, deltas, edits_applied, skipped)

    # restore original state (non-destructive baseline)
    som.set_state(base_state)
    return final_bins, stats


# -------------------------------------------------------------------
# 2) Greedy Pointer baseline (global argmax at each step)
# -------------------------------------------------------------------

def greedy_pointer(
    som: "SOM",
    steps: int = 5_000,
    node_shortlist: Optional[int] = None,   # e.g., 1024; if None, scan all nodes
    top_M_bins: int = 3,
    allow_negative: bool = False,
    cooldown_steps: int = 12,
    seed: Optional[int] = 0,
) -> Tuple[np.ndarray, RunStats]:
    """
    At each step:
      - choose node i* with highest max_b Δ(i->b) (optionally from a shortlist)
      - set bin to argmax Δ for that node if Δ>0 (or if allow_negative=True)
      - cooldown node if realized Δ <= 0
    """
    rng = np.random.default_rng(seed)
    H, W = som.grid_shape
    N = H * W
    K = som.num_bins

    base_state = som.get_state()
    trace_hist, deltas = [], []
    edits_applied = 0
    skipped = 0
    cool_until = np.zeros((N,), dtype=np.int64)

    t = 0
    trace_hist.append(som.total_reward_fast())

    node_indices = np.arange(N, dtype=np.int64)

    while t < steps:
        # shortlist candidates (optional)
        candidates = node_indices
        if node_shortlist is not None and node_shortlist < N:
            # quick-and-dirty saliency: sample a pool, pick top by a cheap proxy
            pool = rng.choice(node_indices, size=min(4*node_shortlist, N), replace=False)
            # approximate saliency = max_b Δ for each pool node
            pool_scores = []
            for node in pool:
                if cool_until[node] > t:  # skip cooled nodes
                    pool_scores.append(-np.inf)
                    continue
                dvec = som.estimate_delta_trace_bins_for_node(int(node))
                dvec[int(som.get_assigned_bins(grid=False)[node])] = -np.inf
                pool_scores.append(np.max(dvec))
            pool_scores = np.array(pool_scores)
            top_idx = np.argpartition(pool_scores, -node_shortlist)[-node_shortlist:]
            candidates = pool[top_idx]

        # pick best candidate
        best_node = None; best_bin = None; best_delta = -np.inf
        for node in candidates:
            node = int(node)
            if cool_until[node] > t:
                continue
            dvec = som.estimate_delta_trace_bins_for_node(node)
            cur_bin = int(som.get_assigned_bins(grid=False)[node])
            dvec[cur_bin] = -np.inf
            b_star = int(np.argmax(dvec))
            d_star = float(dvec[b_star])
            if d_star > best_delta:
                best_delta, best_node, best_bin = d_star, node, b_star

        if best_node is None:
            # all candidates cooled; advance time and continue
            deltas.append(0.0); skipped += 1; t += 1
            trace_hist.append(som.total_reward_fast())
            continue

        if (best_delta > 0.0) or allow_negative:
            r0 = som.total_reward_fast()
            x, y = divmod(best_node, W)
            som.set_node_bin(x, y, int(best_bin))
            r1 = som.total_reward_fast()
            delta = float(r1 - r0)
            deltas.append(delta)
            edits_applied += 1
            if delta <= 0.0:
                cool_until[best_node] = t + cooldown_steps
        else:
            deltas.append(0.0)
            cool_until[best_node] = t + cooldown_steps
            skipped += 1

        t += 1
        trace_hist.append(som.total_reward_fast())

    final_bins = som.get_state()
    stats = _summarize(trace_hist, deltas, edits_applied, skipped)

    # restore state
    som.set_state(base_state)
    return final_bins, stats


# -------------------------------------------------------------------
# 3) Progressive short-episode scheduler (for RL or greedy loops)
# -------------------------------------------------------------------

@dataclass
class HorizonSchedule:
    init_steps: int = 200
    step_growth: int = 100             # +100 steps after each promotion
    max_steps: int = 5_000
    promote_min_pos_rate: float = 0.60 # need >=60% positive Δ in last block to promote
    eval_block: int = 200              # evaluate over the last 'eval_block' steps


class ProgressiveHorizon:
    """
    Simple episodic progression: start with small horizon, increase when recent
    ΔTrace-positive rate >= threshold. Works for both greedy baselines and RL loops.
    """
    def __init__(self, schedule: HorizonSchedule):
        self.sched = schedule
        self.cur_steps = schedule.init_steps
        self._recent = []

    def record(self, delta: float):
        self._recent.append(float(delta))
        if len(self._recent) > self.sched.eval_block:
            self._recent.pop(0)

    def on_episode_end(self):
        # decide promotion
        arr = np.array(self._recent, dtype=np.float64)
        if arr.size >= self.sched.eval_block:
            pos_rate = float(np.mean(arr > 0.0))
            if pos_rate >= self.sched.promote_min_pos_rate:
                self.cur_steps = min(self.cur_steps + self.sched.step_growth, self.sched.max_steps)
        # reset recent buffer
        self._recent = []

    def current_horizon(self) -> int:
        return int(self.cur_steps)


# -------------------------------------------------------------------
# 4) Tiny demo / tests (requires a SOM instance)
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Minimal synthetic SOM to smoke-test the baselines quickly.
    rng = np.random.default_rng(0)
    K = 5; H, W = 20, 20; N = H*W
    node_idx_photo = rng.integers(0, N, size=30_000)
    init_bins = rng.integers(0, K, size=N, dtype=np.int32)
    dd = rng.random((K, N)).astype(np.float32) * 100 + 1e-3
    dr = rng.random((K, N)).astype(np.float32) * 10  + 1e-3
    rd = rng.random((K, N)).astype(np.float32) * 10  + 1e-3
    rr = rng.random((K, N)).astype(np.float32) * 50  + 1e-3

    som = SOM(node_idx_photo, init_bins, dd, dr, rd, rr, num_bins=K, seed=42)

    print("== Greedy Conveyor ==")
    _, stats_conv = greedy_conveyor(som, passes=2, order="random", top_M_bins=3, allow_negative=False)
    print(stats_conv.as_dict())

    print("== Greedy Pointer ==")
    _, stats_ptr = greedy_pointer(som, steps=1000, node_shortlist=512, top_M_bins=3, allow_negative=False)
    print(stats_ptr.as_dict())

    # Progressive horizon example (pseudo-episode loop)
    sched = HorizonSchedule(init_steps=200, step_growth=100, max_steps=2000, promote_min_pos_rate=0.6, eval_block=200)
    prog = ProgressiveHorizon(sched)
    for ep in range(5):
        # toy: pretend we ran an episode of prog.current_horizon() steps
        # and we saw these deltas:
        fake_deltas = rng.normal(loc=0.001*(ep+1), scale=0.002, size=prog.current_horizon())
        for d in fake_deltas:
            prog.record(d)
        prog.on_episode_end()
        print(f"Episode {ep}: new horizon = {prog.current_horizon()}")


== Greedy Conveyor ==
{'initial_trace': 1045.639334231845, 'final_trace': 7330.353323116238, 'total_delta': 6284.713988884393, 'steps_taken': 800, 'positive_steps': 483, 'zero_steps': 317, 'negative_steps': 0, 'skipped_steps': 317, 'edits_applied': 483}
== Greedy Pointer ==
{'initial_trace': 1045.639334231845, 'final_trace': 292300.4468438205, 'total_delta': 291254.80750958866, 'steps_taken': 1000, 'positive_steps': 532, 'zero_steps': 468, 'negative_steps': 0, 'skipped_steps': 468, 'edits_applied': 532}
Episode 0: new horizon = 300
Episode 1: new horizon = 400
Episode 2: new horizon = 500
Episode 3: new horizon = 600
Episode 4: new horizon = 700


Now again with real data:

In [6]:
# -------------------------------------------------------------------
# 4) Tiny demo / tests (requires a SOM instance)
# -------------------------------------------------------------------
import os

if __name__ == "__main__":
    # Minimal synthetic SOM to smoke-test the baselines quickly.
    rng = np.random.default_rng(0)
    H, W = 200, 200; N = H*W

    ROOT = "data"
    NPZ = os.path.join(ROOT, "correlations_dl", "npairs_compact.npz")
    INIT = os.path.join(ROOT, "nodes_labels_initial_fc_200_ext.npy")
    NODE = os.path.join(ROOT, "node_indices_fc.npy")  # photometric-only node ids
    num_bins = 11

    print("[load] correlations...")
    npz = np.load(NPZ)
    dd, dr, rd, rr = npz['dd'], npz['dr'], npz['rd'], npz['rr']

    initial_node_bins = np.load(INIT)   # (num_nodes,)
    node_indices = np.load(NODE)        # per-photometric galaxy node id
    num_nodes = initial_node_bins.size

    som = SOM(node_idx_photo, init_bins, dd, dr, rd, rr, num_bins=num_bins, seed=42)

    print("== Greedy Conveyor ==")
    _, stats_conv = greedy_conveyor(som, passes=2, order="random", top_M_bins=3, allow_negative=False)
    print(stats_conv.as_dict())

    print("== Greedy Pointer ==")
    _, stats_ptr = greedy_pointer(som, steps=1000, node_shortlist=512, top_M_bins=3, allow_negative=False)
    print(stats_ptr.as_dict())

    # Progressive horizon example (pseudo-episode loop)
    sched = HorizonSchedule(init_steps=200, step_growth=100, max_steps=2000, promote_min_pos_rate=0.6, eval_block=200)
    prog = ProgressiveHorizon(sched)
    for ep in range(5):
        # toy: pretend we ran an episode of prog.current_horizon() steps
        # and we saw these deltas:
        fake_deltas = rng.normal(loc=0.001*(ep+1), scale=0.002, size=prog.current_horizon())
        for d in fake_deltas:
            prog.record(d)
        prog.on_episode_end()
        print(f"Episode {ep}: new horizon = {prog.current_horizon()}")

[load] correlations...


ValueError: non-broadcastable output operand with shape (11,) doesn't match the broadcast shape (11,11)

### RL

In [5]:
# env_pointer_greedy_bin.py
from __future__ import annotations
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class PointerGreedyBinEnv(gym.Env):
    """
    Action: pick a NODE (0..N-1). The env then
      - computes Δ(i->b) for all absolute bins b with the SOM Δ-predictor
      - picks b* = argmax Δ (greedy)
      - if Δ(b*) > 0: apply it and reward = exact ΔTrace
        else: (by default) do NOT change state, reward = 0, cooldown node
             (optionally 'allow_negative_with_gate' can approve certain negatives)

    Masking: node i is valid iff (max_b Δ(i->b) > 0) and not on cooldown,
             with a tiny ε-unmask for exploration.

    This keeps the action space = N (e.g., 200*200=40k) and leaves bins deterministic.
    """

    metadata = {"render_modes": ["ansi"]}

    def __init__(
        self,
        som,                                   # your SOM with estimate_delta_trace_bins_for_node() and total_reward_fast()
        max_steps: int = 2000,
        cooldown_steps: int = 12,
        epsilon_unmask: float = 0.02,
        allow_negative_with_gate: bool = False,
        gate_neighbors: str = "4",             # only used if allow_negative_with_gate=True
        gate_tau: float = 0.0,                 # minimal positive net 2-step needed to approve a negative first step
        seed: int | None = None,
    ):
        super().__init__()
        self.som = som
        self.H, self.W = som.grid_shape
        self.N = self.H * self.W
        self.K = som.num_bins
        self.max_steps = int(max_steps)
        self.cool_T = int(cooldown_steps)
        self.eps_unmask = float(epsilon_unmask)
        self.allow_negative_with_gate = bool(allow_negative_with_gate)
        self.gate_tau = float(gate_tau)
        self.rng = np.random.default_rng(seed)

        # action = node index
        self.action_space = spaces.Discrete(self.N)
        # observation = H×W×C tensor (same channels as before)
        self._obs_channels = 7
        self.observation_space = spaces.Box(
            low=-10, high=10, shape=(self.H, self.W, self._obs_channels), dtype=np.float32
        )

        # obs cache
        self._norm_counts = max(1.0, float(np.max(self.som.get_node_counts_total(grid=True))))
        self._obs_grid = self._build_obs()

        # episode vars
        self.steps = 0
        self._cool_until = np.zeros(self.N, dtype=np.int64)

        # neighbors for optional 2-step gate
        self._neighbors = self._build_neighbors(mode=gate_neighbors)

    # ---------- obs helpers ----------
    def _build_obs(self) -> np.ndarray:
        A = self.som.get_assigned_bins(grid=True).astype(np.float32)
        P = self.som.get_photoz_bin_mode().astype(np.float32)
        T = self.som.get_truez_bin_mode().astype(np.float32)
        U = self.som.get_node_umatrix().astype(np.float32)
        Nt = self.som.get_node_counts_total(grid=True).astype(np.float32) / self._norm_counts
        Ns = self.som.get_node_counts_spec().astype(np.float32)
        K1 = max(1.0, self.K - 1.0)
        A /= K1; P /= K1; T /= K1
        if np.std(U) > 0:
            U = (U - np.mean(U)) / (np.std(U) + 1e-6)
        D = (A - T).astype(np.float32)
        return np.stack([A, P, T, D, U, Nt, Ns], axis=-1).astype(np.float32)

    def _update_cell_in_obs(self, node_idx: int):
        x = int(node_idx // self.W); y = int(node_idx % self.W)
        K1 = max(1.0, self.K - 1.0)
        bval = float(self.som.get_assigned_bins(grid=True)[x, y]) / K1
        self._obs_grid[x, y, 0] = bval  # A
        self._obs_grid[x, y, 1] = bval  # P (stand-in)
        self._obs_grid[x, y, 3] = bval - self._obs_grid[x, y, 2]  # D

    # ---------- neighbors & gate ----------
    def _build_neighbors(self, mode="4"):
        nbrs = [[] for _ in range(self.N)]
        for i in range(self.N):
            x, y = divmod(i, self.W)
            if mode == "8":
                deltas = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
            else:
                deltas = [(-1,0),(1,0),(0,-1),(0,1)]
            for dx, dy in deltas:
                xx, yy = x + dx, y + dy
                if 0 <= xx < self.H and 0 <= yy < self.W:
                    nbrs[i].append(xx * self.W + yy)
        return [np.array(v, dtype=np.int64) for v in nbrs]

    def _two_step_gate_allows(self, node: int, b_first: int, d_first: float) -> bool:
        """Counterfactual, neighbors-only 2-step lookahead. No state mutation."""
        if d_first >= 0.0:
            return True
        # Build counterfactual diagonals & nphot deltas for applying (node->b_first)
        i = node
        old_i = int(self.som.current_bins[i])
        if b_first == old_i:
            return False
        w_i = float(self.som.node_gal_counts[i])

        # Grab needed diag sums and counts (copy only for touched bins)
        DD = self.som.DD_sum.diagonal().copy()
        DR = self.som.DR_sum.diagonal().copy()
        RD = self.som.RD_sum.diagonal().copy()
        RR = self.som.RR_sum.diagonal().copy()
        nref = self.som._n_ref_proxy  # fixed
        nphot = self.som.galaxy_bin_counts.copy()

        # Apply i->b_first counterfactually to diagonals & counts
        DD[old_i] -= self.som.dd[old_i, i]; DR[old_i] -= self.som.dr[old_i, i]; RD[old_i] -= self.som.rd[old_i, i]; RR[old_i] -= self.som.rr[old_i, i]
        nphot[old_i] -= w_i
        DD[b_first] += self.som.dd[b_first, i]; DR[b_first] += self.som.dr[b_first, i]; RD[b_first] += self.som.rd[b_first, i]; RR[b_first] += self.som.rr[b_first, i]
        nphot[b_first] += w_i

        # Evaluate best neighbor follow-up under this counterfactual
        best_second = -np.inf
        for j in self._neighbors[i]:
            old_j = int(self.som.current_bins[j])
            w_j = float(self.som.node_gal_counts[j])
            if w_j <= 0:
                continue
            # Δ for j->b' uses only diag old_j and b' (counterfactual versions)
            # scan all b'
            for b2 in range(self.K):
                if b2 == old_j:
                    continue
                # previous xi(old_j), xi(b2)
                xi_old_prev = self._xi_diag(DD[old_j], DR[old_j], RD[old_j], RR[old_j], nref[old_j], max(nphot[old_j], 1e-12))
                xi_b2_prev  = self._xi_diag(DD[b2],   DR[b2],   RD[b2],   RR[b2],   nref[b2],   max(nphot[b2],   1e-12))
                # after-move j->b2 deltas
                DD_old_a = DD[old_j] - self.som.dd[old_j, j]; DR_old_a = DR[old_j] - self.som.dr[old_j, j]; RD_old_a = RD[old_j] - self.som.rd[old_j, j]; RR_old_a = RR[old_j] - self.som.rr[old_j, j]
                nphot_old_a = nphot[old_j] - w_j
                DD_b2_a = DD[b2] + self.som.dd[b2, j]; DR_b2_a = DR[b2] + self.som.dr[b2, j]; RD_b2_a = RD[b2] + self.som.rd[b2, j]; RR_b2_a = RR[b2] + self.som.rr[b2, j]
                nphot_b2_a = nphot[b2] + w_j

                if nphot_old_a <= 0 or nphot_b2_a <= 0:
                    d2 = -np.inf
                else:
                    xi_old_a = self._xi_diag(DD_old_a, DR_old_a, RD_old_a, RR_old_a, nref[old_j], nphot_old_a)
                    xi_b2_a  = self._xi_diag(DD_b2_a,  DR_b2_a,  RD_b2_a,  RR_b2_a,  nref[b2],   nphot_b2_a)
                    d2 = (xi_old_a - xi_old_prev) + (xi_b2_a - xi_b2_prev)
                if d2 > best_second:
                    best_second = d2

        return (d_first + best_second) > self.gate_tau

    # reuse SOM's xi function
    def _xi_diag(self, DDjj, DRjj, RDjj, RRjj, nref_j, nphot_j) -> float:
        den = (nref_j * nphot_j)
        DDn = DDjj / den
        DRn = DRjj / (10.0 * den)
        RDn = RDjj / (10.0 * den)
        RRn = RRjj / (100.0 * den)
        if RRn <= 0:
            return 0.0
        return float((DDn - DRn - RDn + RRn) / RRn)

    # ---------- masking ----------
    def valid_action_mask(self) -> np.ndarray:
        mask = np.zeros(self.N, dtype=bool)
        now = self.steps
        # Allow nodes with positive predicted Δ (and not cooled).
        # Also add a tiny ε exploration to randomly unmask a few extras.
        allow = []
        extra = []
        # Sample a subset for speed if N is huge (optional); here we just loop (40k is OK).
        for i in range(self.N):
            if self._cool_until[i] > now:
                continue
            dvec = self.som.estimate_delta_trace_bins_for_node(i)
            dvec[int(self.som.current_bins[i])] = -np.inf
            dmax = float(np.max(dvec))
            if dmax > 0.0:
                allow.append(i)
            else:
                extra.append(i)
        mask[allow] = True
        # ε-unmask a small random set from 'extra' to keep some exploration alive
        if self.eps_unmask > 0 and extra:
            k = max(1, int(self.eps_unmask * len(extra)))
            pick = self.rng.choice(np.array(extra, dtype=np.int64), size=k, replace=False)
            mask[pick] = True
        # Ensure at least one valid action exists
        if not mask.any():
            # fall back: unmask the single best node by predicted Δ (even if <=0)
            best_node = None; best_score = -np.inf
            for i in range(self.N):
                if self._cool_until[i] > now:
                    continue
                dvec = self.som.estimate_delta_trace_bins_for_node(i)
                dvec[int(self.som.current_bins[i])] = -np.inf
                dmax = float(np.max(dvec))
                if dmax > best_score:
                    best_score = dmax; best_node = i
            if best_node is not None:
                mask[best_node] = True
        return mask

    # ---------- gym api ----------
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.som.reset_to_initial()
        if options and options.get("randomize", 0) > 0:
            self.som.randomize_state(options["randomize"])
        self._obs_grid = self._build_obs()
        self.steps = 0
        self._cool_until[:] = 0
        return self._obs_grid, {"trace": float(self.som.total_reward_fast())}

    def step(self, action: int):
        self.steps += 1
        node = int(action)
        node = max(0, min(self.N - 1, node))
        info = {}
        reward = 0.0
        terminated = False
        truncated = self.steps >= self.max_steps

        # skip if cooled
        if self._cool_until[node] > self.steps:
            return self._obs_grid, reward, terminated, truncated, info

        # greedy bin for this node
        dvec = self.som.estimate_delta_trace_bins_for_node(node)
        cur = int(self.som.current_bins[node])
        dvec[cur] = -np.inf
        b_star = int(np.argmax(dvec))
        d_pred = float(dvec[b_star])

        apply_move = False
        if d_pred > 0.0:
            apply_move = True
        elif self.allow_negative_with_gate:
            apply_move = self._two_step_gate_allows(node, b_star, d_pred)

        if apply_move:
            r0 = self.som.total_reward_fast()
            x, y = divmod(node, self.W)
            self.som.set_node_bin(x, y, b_star)
            r1 = self.som.total_reward_fast()
            reward = float(r1 - r0)
            self._update_cell_in_obs(node)
            # cooldown only if realized Δ ≤ 0 (should be rare if d_pred>0)
            if reward <= 0.0:
                self._cool_until[node] = self.steps + self.cool_T
        else:
            # no change; cooldown this node briefly so we don't hammer it
            self._cool_until[node] = self.steps + self.cool_T

        info["chosen_node"] = node
        info["chosen_bin"] = int(b_star)
        info["pred_delta"] = float(d_pred)
        info["trace"] = float(self.som.total_reward_fast())
        return self._obs_grid, reward, terminated, truncated, info

    def render(self):
        return f"steps={self.steps}, trace={self.som.total_reward_fast():.6f}"


In [ ]:
# pip install sb3-contrib stable-baselines3
from sb3_contrib.ppo_mask import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from env_pointer_greedy_bin import PointerGreedyBinEnv
from som_rl import SOM  # your SOM class with Δ-predictor

som = SOM(...)

env = PointerGreedyBinEnv(som, max_steps=500, cooldown_steps=12, epsilon_unmask=0.02,
                          allow_negative_with_gate=False)

def mask_fn(e): return e.valid_action_mask()
env_masked = ActionMasker(env, mask_fn)

model = MaskablePPO("CnnPolicy", env_masked, verbose=1, n_steps=2048, batch_size=512,
                    gamma=1.0, ent_coef=0.01, learning_rate=3e-4)
model.learn(total_timesteps=200_000)
